# Task 1 — LoRA adaptation of small generalist VLMs

**Owner:** Shanmuga Perumal Balamurugan  
**Hypothesis (H1):** LoRA-adapted 3B generalists approach specialist parity on ScreenSpot-V2 (within ±3 pp) but retain a 10-15 pp gap on ScreenSpot-Pro.

**Variations**
1. Qwen2.5-VL-3B zero-shot (baseline from notebook 00)
2. Qwen2.5-VL-3B + LoRA (r=8, 16, 32, 64)
3. PaliGemma-3B + LoRA (r=8, 16, 32, 64) — secondary backbone

**Deeper analysis:** rank ablation (accuracy vs. trainable params vs. peak VRAM) + error analysis by UI type.

In [ ]:
from ais5.adapt import LoRAConfig, TrainingArgs, run_lora_training
from ais5.utils import set_global_seed

set_global_seed(42)

## 1. Train one rank end-to-end (r=8) on a 5K subset

Verify the training loop converges before scaling to the full sweep.

In [ ]:
lora = LoRAConfig(r=8, alpha=16, backbone='qwen2.5-vl')
args = TrainingArgs(
    output_dir='checkpoints/qwen-r8-5k',
    train_subset_size=5_000,
    num_train_epochs=1.0,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    bf16=True,
)
adapter_dir = run_lora_training('qwen2.5-vl-3b', lora, args)
print(adapter_dir)

## 2. Evaluate the adapter on ScreenSpot-V2 + Pro

In [ ]:
from ais5.data import load_benchmark
from ais5.eval import evaluate_model
from ais5.models import get_model

model = get_model('qwen2.5-vl-3b', peft_adapter=str(adapter_dir))
results = {}
for bench in ['screenspot-v2', 'screenspot-pro', 'osworld-g']:
    run = evaluate_model(model, list(load_benchmark(bench)), benchmark=bench, limit=200)
    results[bench] = run.accuracy
    print(f'{bench:18s} acc={run.accuracy:.4f} (n={len(run.results)})')

## 3. Rank sweep (full 50K subset)

Re-run the cell above for r ∈ {8, 16, 32, 64} and aggregate. Save trainable-param counts to plot the rank-ablation curve.

In [ ]:
import pandas as pd

# Manual aggregation pattern — populate as each rank finishes.
rows = []
for rank in [8, 16, 32, 64]:
    rows.append({
        'rank': rank,
        'screenspot_v2': None,
        'screenspot_pro': None,
        'osworld_g': None,
        'trainable_params_M': None,
        'peak_vram_gb': None,
    })
df = pd.DataFrame(rows)
df

## 4. Error analysis by UI type

Where does LoRA help most? Compare zero-shot vs. r=8 vs. r=64 broken down by UI type (web/desktop/mobile).

In [ ]:
from ais5.eval.breakdown import by_ui_type

by_ui_type(run.results)